# HTTP Requests with the `requests` Library

The `requests` library makes HTTP requests simple and human-friendly.

Install: `pip install requests`

We'll use **JSONPlaceholder** (`https://jsonplaceholder.typicode.com`) — a free fake REST API for testing.

In [ ]:
import requests

print(f'requests version: {requests.__version__}')

## 1. GET Request

GET is used to **retrieve** data from a server.

In [ ]:
import requests

response = requests.get('https://jsonplaceholder.typicode.com/posts/1')

# Response properties
print(f'Status code: {response.status_code}')   # 200
print(f'Content-Type: {response.headers["Content-Type"]}')
print(f'Encoding: {response.encoding}')

# Parse JSON response
data = response.json()
print(f'\nPost ID: {data["id"]}')
print(f'Title: {data["title"]}')
print(f'Body: {data["body"][:80]}...')

In [ ]:
# GET with query parameters
params = {'userId': 1}  # filter posts by user
response = requests.get('https://jsonplaceholder.typicode.com/posts', params=params)

posts = response.json()
print(f'Posts by user 1: {len(posts)}')
for post in posts[:3]:
    print(f"  [{post['id']}] {post['title'][:50]}")

## 2. Response Object

In [ ]:
import requests

r = requests.get('https://jsonplaceholder.typicode.com/users/1')

print(f'Status: {r.status_code}')           # 200
print(f'OK: {r.ok}')                        # True if 200-299
print(f'URL: {r.url}')                      # final URL (after redirects)
print(f'Elapsed: {r.elapsed}')              # time taken

# Access response content
print('\nText (raw string):')               
print(r.text[:100])

print('\nJSON (dict):')
user = r.json()
print(f"  Name: {user['name']}")
print(f"  Email: {user['email']}")
print(f"  Company: {user['company']['name']}")

## 3. POST Request — Sending Data

In [ ]:
import requests

# POST sends data to create a resource
new_post = {
    'title': 'My New Post',
    'body': 'This is the post content.',
    'userId': 1
}

response = requests.post(
    'https://jsonplaceholder.typicode.com/posts',
    json=new_post  # automatically sets Content-Type: application/json
)

print(f'Status: {response.status_code}')  # 201 Created
created = response.json()
print(f'Created post ID: {created["id"]}')
print(f'Title: {created["title"]}')

## 4. PUT and DELETE Requests

In [ ]:
import requests

# PUT — update an existing resource
updated_post = {'id': 1, 'title': 'Updated Title', 'body': 'Updated body', 'userId': 1}
r = requests.put('https://jsonplaceholder.typicode.com/posts/1', json=updated_post)
print(f'PUT status: {r.status_code}')  # 200
print(f'Updated title: {r.json()["title"]}')

# PATCH — partial update
r = requests.patch('https://jsonplaceholder.typicode.com/posts/1',
                   json={'title': 'Patched Title'})
print(f'PATCH status: {r.status_code}')
print(f'Patched title: {r.json()["title"]}')

# DELETE
r = requests.delete('https://jsonplaceholder.typicode.com/posts/1')
print(f'DELETE status: {r.status_code}')  # 200

## 5. Headers, Timeouts, and Error Handling

In [ ]:
import requests

# Custom headers
headers = {
    'User-Agent': 'MyApp/1.0',
    'Accept': 'application/json',
    'Authorization': 'Bearer fake_token_here'
}

# Timeout prevents hanging forever
try:
    r = requests.get(
        'https://jsonplaceholder.typicode.com/posts/1',
        headers=headers,
        timeout=5  # seconds
    )
    r.raise_for_status()  # raises HTTPError for 4xx/5xx
    print(f'Success: {r.status_code}')
    
except requests.exceptions.Timeout:
    print('Request timed out!')
except requests.exceptions.HTTPError as e:
    print(f'HTTP error: {e}')
except requests.exceptions.ConnectionError:
    print('Connection failed!')
except requests.exceptions.RequestException as e:
    print(f'Request failed: {e}')

## 6. Session — Reuse Connection + Headers

In [ ]:
import requests

# Session reuses TCP connection and shares headers/cookies
with requests.Session() as session:
    # Set headers once — applied to all requests
    session.headers.update({
        'User-Agent': 'MyApp/1.0',
        'Accept': 'application/json'
    })
    
    # Make multiple requests
    users_r = session.get('https://jsonplaceholder.typicode.com/users')
    posts_r = session.get('https://jsonplaceholder.typicode.com/posts?_limit=5')
    
    users = users_r.json()
    posts = posts_r.json()
    
    print(f'Fetched {len(users)} users and {len(posts)} posts')
    print(f'First user: {users[0]["name"]}')
    print(f'First post: {posts[0]["title"][:40]}')

## Mini-Project: JSONPlaceholder API Wrapper

In [ ]:
import requests

class JSONPlaceholderAPI:
    """Simple wrapper for the JSONPlaceholder API."""
    
    BASE_URL = 'https://jsonplaceholder.typicode.com'
    
    def __init__(self):
        self.session = requests.Session()
        self.session.headers.update({'Accept': 'application/json'})
    
    def _get(self, endpoint, params=None):
        r = self.session.get(f'{self.BASE_URL}{endpoint}', params=params, timeout=10)
        r.raise_for_status()
        return r.json()
    
    def get_user(self, user_id):
        return self._get(f'/users/{user_id}')
    
    def get_users(self):
        return self._get('/users')
    
    def get_posts_by_user(self, user_id):
        return self._get('/posts', params={'userId': user_id})
    
    def get_todos_by_user(self, user_id):
        return self._get('/todos', params={'userId': user_id})
    
    def create_post(self, user_id, title, body):
        r = self.session.post(
            f'{self.BASE_URL}/posts',
            json={'userId': user_id, 'title': title, 'body': body},
            timeout=10
        )
        r.raise_for_status()
        return r.json()
    
    def user_summary(self, user_id):
        user = self.get_user(user_id)
        posts = self.get_posts_by_user(user_id)
        todos = self.get_todos_by_user(user_id)
        completed = sum(1 for t in todos if t['completed'])
        
        print(f'User: {user["name"]} ({user["email"]})')
        print(f'Company: {user["company"]["name"]}')
        print(f'Posts: {len(posts)}')
        print(f'Todos: {len(todos)} ({completed} completed, {len(todos)-completed} pending)')
        print(f'Recent post: {posts[0]["title"][:50] if posts else "none"}')


api = JSONPlaceholderAPI()
print('=== User Summary ===')
api.user_summary(1)

print('\n=== New Post ===')
post = api.create_post(1, 'My Test Post', 'This is my content')
print(f'Created: [{post["id"]}] {post["title"]}')